# Anova and Tukey HSD Tests
## One way or two way ANOVA specified in formula string

In [40]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.anova import anova_lm
from scipy.stats import levene  # Corrected import
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# ---------------------------
# Step 1: Define File Paths
# ---------------------------

# Path to the aggregated bootstrapped results CSV
csv_path = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\aggregated_bootstrap_results.csv"

# Output directory for ANOVA and Tukey's HSD results
anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors_Interaction"
formula = 'Coefficient_Estimate ~ C(Fire_Type) + C(Predictor) + C(Fire_Type):C(Predictor)'

# One way ANOVA formula
# anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\One-way_ANOVA_Results"
# formula = 'Coefficient_Estimate ~ C(Fire_Type)'

# Two way ANOVA formula
# anova_results_dir = r"C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results"
# formula = 'Coefficient_Estimate ~ C(Fire_Type) + C(Predictor)'

# Create subdirectories
os.makedirs(anova_results_dir, exist_ok=True)
os.makedirs(os.path.join(anova_results_dir, "Individual Cases"), exist_ok=True)

# ---------------------------
# Step 2: Load and Prepare Data
# ---------------------------

# Load the data from the CSV file
try:
    df = pd.read_csv(csv_path)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file at '{csv_path}' was not found.")
    exit()

# Select relevant columns
required_columns = ['Region', 'term', 'estimate', 'bootstrap_rep', 'Segmentation Interval', 'Dependent Variable', 'p.value']

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    print(f"Error: The following required columns are missing in the CSV: {missing_columns}")
    exit()

anova_df = df[required_columns].copy()

# Remove rows with p.value greater than 0.05
anova_df = anova_df[anova_df['p.value'] <= 0.05]

# Rename columns for clarity
anova_df = anova_df.rename(columns={
    'Region': 'Fire_Type',
    'term': 'Predictor',
    'estimate': 'Coefficient_Estimate',
    'Segmentation Interval': 'Segmentation_Interval',
    'Dependent Variable': 'Dependent_Variable'
})

# Convert categorical variables to 'category' dtype
anova_df['Fire_Type'] = anova_df['Fire_Type'].astype('category')
anova_df['Predictor'] = anova_df['Predictor'].astype('category')
anova_df['Segmentation_Interval'] = anova_df['Segmentation_Interval'].astype('category')
anova_df['Dependent_Variable'] = anova_df['Dependent_Variable'].astype('category')

# Remove the '(Intercept)' Predictor to avoid multicollinearity
anova_df = anova_df[anova_df['Predictor'] != '(Intercept)']

# ---------------------------
# Step 3: Identify and Remove Problematic Predictors
# ---------------------------

# a. Identify Predictors with Only One Fire Type
single_fire_predictors = anova_df.groupby('Predictor')['Fire_Type'].nunique()
single_fire_predictors = single_fire_predictors[single_fire_predictors < 2].index.tolist()

print("\nPredictors with only one Fire Type:")
print(single_fire_predictors)

# b. Remove Predictors with Only One Fire Type
anova_df = anova_df[~anova_df['Predictor'].isin(single_fire_predictors)]
print(f"\nRemoved {len(single_fire_predictors)} predictors with single Fire Type.")

# c. Identify Predictors with Zero Variance
zero_variance_predictors = anova_df.groupby('Predictor')['Coefficient_Estimate'].var()
zero_variance_predictors = zero_variance_predictors[zero_variance_predictors == 0].index.tolist()

print("\nPredictors with zero variance in Coefficient Estimates:")
print(zero_variance_predictors)

# d. Remove Predictors with Zero Variance
anova_df = anova_df[~anova_df['Predictor'].isin(zero_variance_predictors)]
print(f"\nRemoved {len(zero_variance_predictors)} predictors with zero variance.")

# Save the cleaned dataframe
cleaned_csv_path = os.path.join(anova_results_dir, "cleaned_bootstrap_results.csv")
anova_df.to_csv(cleaned_csv_path, index=False)

# ---------------------------
# Step 4: Define Functions
# ---------------------------

def perform_two_way_anova(subset, segmentation, dependent_var, formula):
    """
    Performs Two-Way ANOVA on the given subset of data.
    
    Parameters:
    - subset (DataFrame): Subset of the data for a specific combination.
    - segmentation (str): Segmentation Interval.
    - dependent_var (str): Dependent Variable.
    
    Returns:
    - anova_table (DataFrame): ANOVA results.
    - model (RegressionResultsWrapper): Fitted OLS model.
    """
    # Two way ANOVA with segmentation interval and fire effects:
    # 'Coefficient_Estimate ~ C(Region) + C(Segmentation_Interval)'
    
    try:
        model = ols(formula, data=subset).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        return anova_table, model
    except Exception as e:
        print(f"Error performing Two-Way ANOVA for '{segmentation}' / '{dependent_var}': {e}")
        return None, None

def perform_tukey_hsd_within_predictor(subset, predictor, segmentation, dependent_var):
    """
    Performs Tukey's HSD test comparing Fire Types within a specific Predictor.
    
    Parameters:
    - subset (DataFrame): Subset of the data for a specific combination.
    - predictor (str): The Predictor variable.
    - segmentation (str): Segmentation Interval.
    - dependent_var (str): Dependent Variable.
    
    Returns:
    - tukey_df (DataFrame): Tukey's HSD results.
    """
    # Subset data for the specific predictor
    predictor_subset = subset[subset['Predictor'] == predictor]
    
    # Ensure there are exactly two groups: "Bennett" and "ET"
    fire_types = predictor_subset['Fire_Type'].unique()
    if len(fire_types) != 2:
        print(f"Skipping Tukey's HSD for Predictor '{predictor}' - Expected 2 Fire Types, found {len(fire_types)}")
        return None
    
    # Perform Tukey's HSD
    try:
        tukey = pairwise_tukeyhsd(endog=predictor_subset['Coefficient_Estimate'],
                                  groups=predictor_subset['Fire_Type'],
                                  alpha=0.05)
        
        # Convert Tukey results to DataFrame
        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
        tukey_df['Segmentation_Interval'] = segmentation
        tukey_df['Dependent_Variable'] = dependent_var
        tukey_df['Predictor'] = predictor
        return tukey_df
    except Exception as e:
        print(f"Error performing Tukey's HSD for Predictor '{predictor}': {e}")
        return None

# ---------------------------
# Step 5: Perform Two-Way ANOVA and Tukey's HSD
# ---------------------------

# Get unique combinations of Segmentation Interval and Dependent Variable
combinations = anova_df[['Segmentation_Interval', 'Dependent_Variable']].drop_duplicates()

# Initialize lists to collect master results
master_anova_results = []
master_tukey_results = []

# Iterate through each combination
for index, row in combinations.iterrows():
    segmentation = row['Segmentation_Interval']
    dependent_var = row['Dependent_Variable']
    
    print(f"\nPerforming Two-Way ANOVA for Segmentation Interval: '{segmentation}' and Dependent Variable: '{dependent_var}'")
    
    # Subset the data for the current combination
    subset = anova_df[
        (anova_df['Segmentation_Interval'] == segmentation) &
        (anova_df['Dependent_Variable'] == dependent_var)
    ]
    
    # Data sufficiency checks
    fire_types = subset['Fire_Type'].unique()
    print(f"Fire Types: {fire_types}")
    if len(fire_types) < 2:
        print(f"Skipping Two-Way ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Regiom Types.")
        continue
    
    predictors = subset['Predictor'].unique()
    if len(predictors) < 2:
        print(f"Skipping Two-Way ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Predictors.")
        continue
    
    bootstrap_reps = subset['bootstrap_rep'].unique()
    if len(bootstrap_reps) < 2:
        print(f"Skipping Two-Way ANOVA for '{segmentation}' / '{dependent_var}' - Not enough Bootstrap Replications.")
        continue
    
    # Perform Two-Way ANOVA
    anova_results = perform_two_way_anova(subset, segmentation, dependent_var, formula)
    if anova_results is None:
        print(f"Skipping further analysis for '{segmentation}' / '{dependent_var}' due to ANOVA errors.")
        continue
    
    anova_table, model = anova_results
    print("\nTwo-Way ANOVA Table:")
    print(anova_table)
    
    # Save ANOVA table to CSV
    anova_filename = os.path.join(anova_results_dir, "Individual Cases", f"Two-Way_ANOVA_{segmentation}_{dependent_var}.csv")
    anova_table.to_csv(anova_filename)
    print(f"Two-Way ANOVA results saved to '{anova_filename}'")
    
    # Optional: Calculate and store effect sizes (e.g., Eta Squared)
    anova_table = anova_table.reset_index()
    anova_table['Segmentation_Interval'] = segmentation
    anova_table['Dependent_Variable'] = dependent_var
    anova_table['Eta_Squared'] = anova_table['sum_sq'] / anova_table['sum_sq'].sum()
    master_anova_results.append(anova_table)
    
    # Since the model does not include interaction, skip interaction checks
    print("No interaction term in the model. Tukey's HSD test not performed.")
    
    # If you still want to perform Tukey's HSD without interaction:
    print("Performing Tukey's HSD test within each Predictor...")
    tukey_results = []
    for predictor in predictors:
        tukey_df = perform_tukey_hsd_within_predictor(subset, predictor, segmentation, dependent_var)
        if tukey_df is not None:
            tukey_results.append(tukey_df)
    
    if tukey_results:
        master_tukey_df = pd.concat(tukey_results, ignore_index=True)
        master_tukey_results.append(master_tukey_df)
        tukey_filename = os.path.join(anova_results_dir, "Individual Cases", f"Tukey_HSD_{segmentation}_{dependent_var}.csv")
        master_tukey_df.to_csv(tukey_filename, index=False)
        print(f"Tukey's HSD results saved to '{tukey_filename}'")
    else:
        print("No Tukey's HSD results to save.")

# ---------------------------
# Step 6: Consolidate ANOVA and Tukey's HSD Results
# ---------------------------

# Consolidate Two-Way ANOVA results into a master CSV
if master_anova_results:
    master_anova_df = pd.concat(master_anova_results, ignore_index=True)
    
    # Save master ANOVA results
    master_anova_filename = os.path.join(anova_results_dir, "Aggregated_TwoWay_ANOVA_Results.csv")
    master_anova_df.to_csv(master_anova_filename, index=False)
    print(f"\nMaster Two-Way ANOVA results saved to '{master_anova_filename}'")
else:
    print("\nNo Two-Way ANOVA results to consolidate.")

# Consolidate Tukey's HSD results into a master CSV
if master_tukey_results:
    master_tukey_df = pd.concat(master_tukey_results, ignore_index=True)
    master_tukey_filename = os.path.join(anova_results_dir, "Aggregated_Tukey_HSD_Results.csv")
    master_tukey_df.to_csv(master_tukey_filename, index=False)
    print(f"Master Tukey's HSD results saved to '{master_tukey_filename}'")
else:
    print("\nNo Tukey's HSD results to consolidate.")


Data loaded successfully.

First few rows of the data:
                          term  estimate  std.error  statistic   p.value  \
0                  (Intercept) -3.321011   0.986805  -3.365419  0.000764   
1  ch_central.slope.difference -0.173531   0.072937  -2.379174  0.017351   
2          ch_curvature.median  0.293623   0.111948   2.622842  0.008720   
3              ch_stream.power  0.238397   0.093819   2.541022  0.011053   
4              ch_valley_width  0.353803   0.113317   3.122225  0.001795   

   conf.low  conf.high  bootstrap_rep     Fire Segmentation Interval  \
0 -5.255112  -1.386910              1  Bennett         Segmented 10m   
1 -0.316485  -0.030576              1  Bennett         Segmented 10m   
2  0.074208   0.513038              1  Bennett         Segmented 10m   
3  0.054514   0.422279              1  Bennett         Segmented 10m   
4  0.131705   0.575901              1  Bennett         Segmented 10m   

       Dependent Variable  
0  lidar_erosion_logtrans  

c:\ProgramData\miniconda3\envs\valley_bottom_env\lib\site-packages\statsmodels\base\model.py:1871: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 41, but rank is 2
  warnings.warn('covariance of constraints does not have full '



Two-Way ANOVA Table:
                  sum_sq      df         F    PR(>F)
C(Fire_Type)    0.312389     1.0  3.473625  0.062398
C(Predictor)    0.460695    41.0  0.124944  0.882548
Residual      599.125581  6662.0       NaN       NaN
Two-Way ANOVA results saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors\Individual Cases\Two-Way_ANOVA_Segmented 10m_lidar_erosion_logtrans.csv'
No interaction term in the model. Tukey's HSD test not performed.
Performing Tukey's HSD test within each Predictor...
Skipping Tukey's HSD for Predictor 'hs_extended.dnbr.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_channel.width.over.valley.width' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_slope.over.width.central.diff' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'hs_aspect.eastness.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Pred

c:\ProgramData\miniconda3\envs\valley_bottom_env\lib\site-packages\statsmodels\base\model.py:1871: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 41, but rank is 1
  warnings.warn('covariance of constraints does not have full '



Two-Way ANOVA Table:
                  sum_sq      df          F        PR(>F)
C(Fire_Type)    7.143585     1.0  65.009276  8.490426e-16
C(Predictor)    1.673297    41.0   0.371406  5.422557e-01
Residual      914.907683  8326.0        NaN           NaN
Two-Way ANOVA results saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors\Individual Cases\Two-Way_ANOVA_Segmented 10m_sfm_deposition_logtrans.csv'
No interaction term in the model. Tukey's HSD test not performed.
Performing Tukey's HSD test within each Predictor...
Skipping Tukey's HSD for Predictor 'hs_extended.dnbr.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_curvature.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ws_aspect.eastness_mean' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_stream.power.central.diff' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Pred

...


Two-Way ANOVA Table:
                  sum_sq      df            F    PR(>F)
C(Fire_Type)   54.376490     1.0  1689.349507  0.000000
C(Predictor)    0.092750    41.0     0.070281  0.975819
Residual      291.492965  9056.0          NaN       NaN
Two-Way ANOVA results saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors\Individual Cases\Two-Way_ANOVA_Segmented 5m_sfm_deposition_logtrans.csv'
No interaction term in the model. Tukey's HSD test not performed.
Performing Tukey's HSD test within each Predictor...
Skipping Tukey's HSD for Predictor 'hs_aspect.northness.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_slope.over.width.central.diff' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_channel.width.over.valley.width' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_stream.power' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for

c:\ProgramData\miniconda3\envs\valley_bottom_env\lib\site-packages\statsmodels\base\model.py:1871: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 41, but rank is 2
  warnings.warn('covariance of constraints does not have full '



Two-Way ANOVA Table:
                  sum_sq       df            F         PR(>F)
C(Fire_Type)   85.259371      1.0  1424.377113  2.288668e-296
C(Predictor)    2.043879     41.0     0.832826   4.348412e-01
Residual      804.721565  13444.0          NaN            NaN
Two-Way ANOVA results saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors\Individual Cases\Two-Way_ANOVA_Segmented 5m_sfm_erosion_logtrans.csv'
No interaction term in the model. Tukey's HSD test not performed.
Performing Tukey's HSD test within each Predictor...
Skipping Tukey's HSD for Predictor 'hs_aspect.northness.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_change.in.slope.over.width' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ws_ndvi.min_mean' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ws_slope.eastness_mean' - Expected 2 Fire Types, found 1
Tukey's HSD resul

c:\ProgramData\miniconda3\envs\valley_bottom_env\lib\site-packages\statsmodels\base\model.py:1871: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 41, but rank is 1
  warnings.warn('covariance of constraints does not have full '



Two-Way ANOVA Table:
                  sum_sq      df           F         PR(>F)
C(Fire_Type)   96.309486     1.0  764.598588  1.288411e-160
C(Predictor)    1.083046    41.0    0.209714   6.470044e-01
Residual      979.975388  7780.0         NaN            NaN
Two-Way ANOVA results saved to 'C:\Users\alextd\Documents\GitHub\ssn2-erosion-deposition-etf-cp\Bootstrap Summary\Two-way_ANOVA_Results_Predictors\Individual Cases\Two-Way_ANOVA_Segmented 5m_sfm_net_logtrans.csv'
No interaction term in the model. Tukey's HSD test not performed.
Performing Tukey's HSD test within each Predictor...
Skipping Tukey's HSD for Predictor 'hs_aspect.northness.median' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ws_aspect.eastness_mean' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_slope.over.width.central.diff' - Expected 2 Fire Types, found 1
Skipping Tukey's HSD for Predictor 'ch_channel.width.over.valley.width' - Expected 2 Fire Types, found 1
Skippin